# 01 · Train a correction starter / 교정기 학습

**목표:** 공개 Train 472개만 사용해 관측된 어절 치환을 학습하고, Eval 입력으로 예측 CSV를 만듭니다.
Python 3.12 이상, CPU, 표준 라이브러리만 사용합니다. Colab에서도 실행할 수 있습니다.

이 모델은 음성을 사용하지 않는 간단한 text-only 기준점입니다. LLM 미세조정 예제는 아니며 성능 개선을 보장하지 않습니다.
Copy보다 나빠질 수 있으므로 반드시 02 노트북으로 비교하세요. 오디오 모델은 README의 Whisper/Qwen 실행 안내를 참고하세요.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json
repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_starter.py").is_file()),
            Path.cwd() / "TURST-YOUR-EARS")
if not (repo / "scripts/train_starter.py").is_file():
    subprocess.run(["git", "clone", "https://github.com/jhparktime/TURST-YOUR-EARS.git", str(repo)], check=True)
os.chdir(repo)
print("Code revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Python:", sys.version.split()[0])
def run(*args):
    subprocess.run([sys.executable, "-m", *map(str, args)], check=True)
data = repo / "datasets/tye-ko-asr-v2"
work = repo / "starter-output"
work.mkdir(exist_ok=True)


## Train만으로 학습
동일 길이의 토큰 치환 중 Train에서 관찰된 치환 중 출현 대비 치환 비율이 90% 이상이고, 두 토큰의 문자 유사도가 0.5 이상인 규칙을 저장합니다.
최소 관측 수는 1입니다. 작은 데이터에서 시작하는 예제이므로 단발성 오류를 외울 수 있습니다. `min-support`를 높이면 적용 규칙 수가 줄어듭니다.
원문 공백과 문장부호는 유지합니다. Eval/Test 정답으로 규칙을 만들거나 정답을 ID별로 저장하지 마세요.


In [ ]:
run("scripts.train_starter", "fit",
    "--inputs", data / "train/inputs.json", "--references", data / "train/references.json",
    "--model", work / "token-model.json", "--min-support", 1, "--min-confidence", 0.9)
model = json.loads((work / "token-model.json").read_text())
print("Learned rules:", len(model["rules"]))
print(json.dumps(dict(list(model["rules"].items())[:10]), ensure_ascii=False, indent=2))


## Eval 추론 — 입력만 사용
이 단계는 `eval/inputs.json`과 학습된 모델만 읽습니다. 정답은 다음 노트북에서 채점할 때 읽습니다.


In [ ]:
run("scripts.train_starter", "predict", "--inputs", data / "eval/inputs.json",
    "--model", work / "token-model.json", "--output", work / "eval-token.csv")
print((work / "eval-token.csv").read_text()[:1000])


## 다음 단계
02_evaluate.ipynb에서 학습 결과와 Copy를 같은 채점기로 비교하세요.
모델을 개선할 때는 학습 데이터, 파라미터, 코드 버전을 기록하고 전체 Eval에 같은 추론 절차를 적용하세요.
음성은 [참가자 ZIP](https://github.com/jhparktime/TURST-YOUR-EARS/releases/tag/dataset-ko-asr-v2)에서 받을 수 있습니다.
